In [31]:
import os
import csv
import h5py
import json
import joblib
import pickle
import numpy as np
import pandas as pd
from pykdtree.kdtree import KDTree
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

Ha_to_eV = 27.21136

def load_json(file_path):
    """Loads data from a JSON file."""
    with open(file_path, 'r') as f:
        return json.load(f)
    
def calculate_formation_energy(energy_dict, atoms_count_array):
    """Calculates formation energy from energy dictionary and atoms count array."""

    assert len(energy_dict) == atoms_count_array.shape[0], "Mismatch in lengths of energy_dict and atoms_count_array"

    energy_array = np.array(list(energy_dict.values()))* Ha_to_eV
    molecules = list(energy_dict.keys())
    reg = LinearRegression(fit_intercept=False)
    reg.fit(atoms_count_array, energy_array)
    predicted_energy = reg.predict(atoms_count_array)
    formation_energy = energy_array - predicted_energy
    return {molecule: energy for molecule, energy in zip(molecules, formation_energy)}

def calculate_target_variable(ccsdt_energy, pbe_energy, atomic_number_dict):
    """Calculates target variable for model training."""
    ccsdt_formation_en = calculate_formation_energy(ccsdt_energy, np.array(list(atomic_number_dict.values())))
    pbe_formation_en = calculate_formation_energy(pbe_energy, np.array(list(atomic_number_dict.values())))

    target_dict = {key: ccsdt_formation_en[key] - pbe_formation_en.get(key, 0) for key in ccsdt_formation_en}

    return target_dict

def get_feature_list_hsmp(max_mcsh_order, step_size, max_r):
    """
    Generates lists of filenames for spin paired HSMP files based on given MCSH parameters.
    Iterates over spherical harmonics orders and cutoff radii.

    :param max_mcsh_order: Maximum order of spherical harmonics.
    :param step_size: Step size for the radial cutoff.
    :param max_r: Maximum radial cutoff.
    :return: list of filenames. 
    """
    hsmp_filenames = []
    for l in range(max_mcsh_order + 1):
        rcut = step_size
        while rcut <= max_r:
            filename = f"HSMP_l_{l}_rcut_{rcut:.6f}_spin_typ_0.csv"
            hsmp_filenames.append(filename)
            rcut += step_size
    return hsmp_filenames

def read_hdf5_data(filepath, num_features, 
                    hsmp_filenames):
    """Read data from an HDF5 file and return a numpy array."""
    with h5py.File(filepath, 'r') as data:
        functional_grp = data["functional_database/PBE"]
        return functional_grp["filtered_feature"][:]

def extract_mcsh_data(data, step_size, mcsh_order, rcut, mcsh_max_order, max_rcut):
    """
    Extracts columns of data corresponding to MCSH order and rcut specified.
    Also includes the first two columns of input data.
    """
    if mcsh_order > mcsh_max_order or rcut > max_rcut:
        raise ValueError("New order or max R must be less than or equal to the original values")

    num_r_values = int(max_rcut/ step_size)
    col_indices = [0, 1]
    for order in range(mcsh_order + 1):
        for r_index in range(num_r_values):
            r_value = (r_index + 1) * step_size
            if r_value > rcut:
                break
            col_index = 2 + order * num_r_values + r_index
            col_indices.append(col_index)
    return data[:, col_indices]

def sort_count_array(count_file, target_dict):
    """Sorts count array based on the order of molecules in CCSDT formation energy dictionary."""
    df = pd.read_csv(count_file, header=None)
    target_dict = {key: target_dict[key] for key in target_dict if key in df[1].values} # Filter out molecules not in target_dict

    df['sort_order'] = df[1].map(lambda x: list(target_dict.keys()).index(x) if x in target_dict else None)
    df_sorted = df.sort_values(by='sort_order').iloc[:, 2:].drop(columns=['sort_order'])
    
    return df_sorted.to_numpy(), np.array(list(target_dict.values())), list(target_dict.keys())

In [51]:
def filter_out_count_arr(fold_target, target_dict=target_dict, final_count_arr=final_count_arr):
    matching_indices = np.where(np.isin(fold_target, list(target_dict.values())))[0]
    filtered_rows = final_count_arr[matching_indices]
    scaler = StandardScaler()
    scaler.fit(filtered_rows)
    return scaler

def scaler_each_fold(mcsh_type, alpha, overall_sig, sys_sig):
    models_path = "/storage/home/hcoda1/0/ssahoo41/cedar_storage/ssahoo41/exact_exchange_work/thesis_datagen/publication_purpose/training_script/models_lasso_pbe_2"
    mcsh_path = os.path.join(models_path, mcsh_type, f"model_all_{overall_sig}_sys_{sys_sig}_lasso", f"alpha_{alpha}")
    scalers = []
    for i in range(5):
        fold_path = os.path.join(mcsh_path, f"{i}_fold_train_true.npy")
        fold_target = np.load(fold_path)
        scaler = filter_out_count_arr(fold_target)
        scalers.append(scaler)
    return scalers

In [52]:
scalers = scaler_each_fold("mcsh_2_rcut_0.5", 0.0001, 0.075, 0.2) 

In [53]:
scalers

[StandardScaler(),
 StandardScaler(),
 StandardScaler(),
 StandardScaler(),
 StandardScaler()]

In [25]:
def partition(data, refdata, max_distance):
    kd_tree = KDTree(refdata,leafsize=6)
    temp_distances, temp_indices = kd_tree.query(data, k=1)
    indices = []
    for i, distance in enumerate(temp_distances): 
        if distance< max_distance: 
            indices.append(temp_indices[i])
        else:
            indices.append(-1) 
    indices = np.array(indices)
    indices_reduced, counts = np.unique(indices, return_counts=True)
    count_arr = np.zeros(len(refdata)+1) 
    for i, index in enumerate(indices_reduced):
        count_arr[index] = counts[i]
    return count_arr

def feature_scaling(feature_arr, mcsh_order, mcsh_step, mcsh_r):
    rcut = np.arange(mcsh_step, mcsh_r + mcsh_step, mcsh_step)
    mcsh_order_list = np.arange(0, mcsh_order + 1, 1)
    index = 2
    for order in mcsh_order_list:
        for rc in rcut:
            feature_arr[:, index] = feature_arr[:, index] * (rc**3)
            index += 1
    return feature_arr

In [26]:
expt_en = {"O3": 1.32000765749999,
          "CCH": 5.4911384108,
          "CH3": 0.7658170968,
           "CH": 5.96348326139999,
           "NH2": 1.4628439994,
           "CN": 4.4009493388,
           "NCCN": 2.76134399449999,
           "O2": -0.0979597583999999,
           "NH": 3.51104703039999,
           "CH2_s1A1d": 3.61791796099999,
           "HCO": 0.0977368273,
           "CH3O": -0.570195164,
           "H2CCO": -1.29323389929999,
           "H3CNH2": -1.77427676999999,
           "OCHCHO": -3.11469143999999,
           "C3H4_C3v": 0.520489227999999,
           "CH2OCH2": -0.705013417499999,
           "CH2NHCH2": -0.373614716999999,
           "C3H6_Cs": -1.73029612699999,
           "CH3CH2NH2": -2.70600651299999,
           "isobutene": -2.76120124,
           "isobutane": -4.53674317599999}

pbe_en_dict = {"O3": -1342.125307,
          "CCH": -337.7393622,
          "CH3": -210.3506509,
           "CH": -173.3655826,
           "NH2": -312.2651631,
           "CN": -441.3785086,
           "NCCN": -889.0324905,
           "O2": -895.7213584,
           "NH": -294.3910489,
           "CH2_s1A1d": -191.3155574,
           "HCO": -627.2142956,
           "CH3O": -659.5206723,
           "H2CCO": -808.2501807,
           "H3CNH2": -526.5383378,
           "OCHCHO": -1257.507578,
           "C3H4_C3v": -553.8814507,
           "CH2OCH2": -840.2878808,
           "CH2NHCH2": -689.0353837,
           "C3H6_Cs": -587.7017953,
           "CH3CH2NH2": -722.8575859,
           "isobutene":-784.0599686 ,
           "isobutane": -817.27613642}

# In the order: H, C, N, O
num_atoms_dict = {"O3": [0, 0, 0, 3],
          "CCH": [1, 2, 0, 0],
          "CH3": [3, 1, 0, 0],
           "CH": [1, 1, 0, 0],
           "NH2": [2, 0, 1, 0],
           "CN": [0, 1, 1, 0],
           "NCCN": [0, 2, 2, 0],
           "O2": [0, 0, 0, 2],
           "NH": [1, 0, 1, 0],
           "CH2_s1A1d": [2, 1, 0, 0],
           "HCO": [1, 1, 0, 1],
           "CH3O": [3, 1, 0, 1],
           "H2CCO": [2, 2, 0, 1],
           "H3CNH2": [5, 1, 1, 0],
           "OCHCHO": [2, 2, 0, 2],
           "C3H4_C3v": [4, 3, 0, 0],
           "CH2OCH2": [4, 2, 0, 1],
           "CH2NHCH2": [5, 2, 1, 0],
           "C3H6_Cs": [6, 3, 0, 0],
           "CH3CH2NH2": [7, 2, 1, 0],
           "isobutene": [8, 4, 0, 0],
           "isobutane": [10, 4, 0, 0]}

In [33]:
json_path = "/storage/home/hcoda1/0/ssahoo41/cedar_storage/ssahoo41/exact_exchange_work/thesis_datagen/publication_purpose/training_script"
ccsdt_file = os.path.join(json_path, "ccsdt_energy.json")
pbe_file = os.path.join(json_path, "pbe_energy.json")
atomic_number_file = os.path.join(json_path, "atoms_count_mat.json")
ccsdt_energy = load_json(ccsdt_file)
pbe_energy = load_json(pbe_file)

atomic_number_dict = load_json(atomic_number_file)
target_dict = calculate_target_variable(ccsdt_energy, pbe_energy, atomic_number_dict)
systems = list(target_dict.keys())

count_file = "/storage/home/hcoda1/0/ssahoo41/cedar_storage/ssahoo41/exact_exchange_work/thesis_datagen/publication_purpose/subsampling_script/partitioning/mcsh_2_rcut_0.5/count_array_overall_0.075_system_0.2.csv"
final_count_arr, target, systems = sort_count_array(count_file, target_dict)

In [4]:
path = "/storage/home/hcoda1/0/ssahoo41/cedar_storage/ssahoo41/exact_exchange_work/thesis_datagen/publication_purpose/training_script/models_lasso_pbe_2/mcsh_2_rcut_4.0/model_all_0.1_sys_0.075_lasso/alpha_0.0001/0_fold_train_true.npy"

In [7]:

filtered_rows = 
train_target = np.load(path)
print(train_target.shape)



(173,)


In [ ]:
# stdscale for count matrix
max_distance_dict = {"mcsh_2_rcut_0.5": {"overall_0.075_sys_0.2": 1.52566166650721},
                    "mcsh_2_rcut_1.0": {"overall_0.075_sys_0.5": 1.54850567974646},
                    "mcsh_2_rcut_1.5": {"overall_0.075_sys_0.5": 1.72909873161774},
                    "mcsh_2_rcut_2.0": {"overall_0.1_sys_0.075": 1.52371145483616},
                    "mcsh_2_rcut_4.0": {"overall_0.1_sys_0.075": 2.09912763358924}}

